# LiveOS OS2 — echtes LoRA-Training auf kostenloser Kaggle-GPU

Dieses Notebook lädt ausschließlich freigegebene OS2-Trainingsbeispiele aus Supabase, trainiert einen Qwen2.5-0.5B-Instruct-Adapter mit LoRA und bewertet ihn anschließend anhand von versteckten OS2-Benchmarks. Die Musterlösungen und Scorer-Regeln bleiben serverseitig.

**Kostenlimit: 0 €.** Kein kostenpflichtiger Anbieter, keine Inference-API, keine GPU-Miete. Kaggle-GPU-Verfügbarkeit und Kontingente sind begrenzt; nach einer beendeten Sitzung startest du das Notebook erneut. Die jeweils letzten Adapter-Gewichte werden in einem privaten Hugging-Face-Repository gespeichert.

Vor dem Start: Kaggle-Einstellungen → Accelerator = GPU und Add-ons → Secrets mit `SUPABASE_SERVICE_ROLE_KEY` und `HF_TOKEN` befüllen. Niemals diese Schlüssel in GitHub oder den Chat kopieren.


In [ ]:
%pip install -q "transformers==4.57.1" "peft==0.17.1" "accelerate==1.10.1" "huggingface_hub==0.35.3" "safetensors==0.6.2"


In [ ]:
import os, json, math, time, requests, torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from datasets import Dataset

BUDGET_EUR = 0
assert BUDGET_EUR == 0, "LiveOS darf nur mit 0-Euro-Budget laufen."
assert torch.cuda.is_available(), "Keine GPU aktiv. Wähle in Kaggle Settings → Accelerator → GPU."
print('GPU:', torch.cuda.get_device_name(0))

secrets = UserSecretsClient()
SUPABASE_URL = os.environ.get('SUPABASE_URL', 'https://twcacjwmahameglodljl.supabase.co').rstrip('/')
SUPABASE_SERVICE_ROLE_KEY = secrets.get_secret('SUPABASE_SERVICE_ROLE_KEY')
HF_TOKEN = secrets.get_secret('HF_TOKEN')
if not SUPABASE_SERVICE_ROLE_KEY or not HF_TOKEN:
    raise RuntimeError('Kaggle Secrets fehlen: SUPABASE_SERVICE_ROLE_KEY und HF_TOKEN müssen hinterlegt sein.')

HEADERS = {
    'apikey': SUPABASE_SERVICE_ROLE_KEY,
    'Authorization': 'Bearer ' + SUPABASE_SERVICE_ROLE_KEY,
    'Content-Type': 'application/json'
}
def rpc(name, payload=None):
    response = requests.post(
        f"{SUPABASE_URL}/rest/v1/rpc/{name}",
        headers=HEADERS,
        json=payload or {},
        timeout=90
    )
    response.raise_for_status()
    return response.json()

print('Verbindung zu Supabase vorbereitet. Schlüssel werden nicht ausgegeben.')


In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
dataset_blob = rpc('liveos_os2_worker_get_dataset')
examples = dataset_blob.get('examples', [])
benchmarks = dataset_blob.get('benchmarks', [])
if len(examples) < 3:
    raise RuntimeError(f'Zu wenige freigegebene OS2-Trainingsbeispiele: {len(examples)}')
if not benchmarks:
    raise RuntimeError('Keine aktiven versteckten OS2-Benchmarks gefunden.')
print(f'{len(examples)} aktive Trainingsbeispiele; {len(benchmarks)} versteckte Benchmarks.')

api = HfApi(token=HF_TOKEN)
identity = api.whoami()
HF_REPO = f"{identity['name']}/liveos-os2-lora"
api.create_repo(repo_id=HF_REPO, repo_type='model', private=True, exist_ok=True)
print('Privates Adapter-Repository:', HF_REPO)

run_id = rpc('liveos_os2_worker_start_run', {
    'p_base_model': BASE_MODEL,
    'p_training_config': {
        'method':'LoRA', 'epochs':2, 'learning_rate':0.0001,
        'gpu':torch.cuda.get_device_name(0), 'budget_eur':0,
        'adapter_repo':HF_REPO
    }
})
print('OS2-Trainingslauf gestartet:', run_id)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
dtype = torch.float16
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=dtype, low_cpu_mem_usage=True,
    token=HF_TOKEN, trust_remote_code=False
).to('cuda')
model.config.use_cache = False

existing_files = api.list_repo_files(repo_id=HF_REPO, repo_type='model')
if 'adapter_config.json' in existing_files:
    print('Vorherige OS2-LoRA-Gewichte gefunden — Training wird darauf fortgesetzt.')
    model = PeftModel.from_pretrained(model, HF_REPO, token=HF_TOKEN, is_trainable=True)
else:
    print('Kein vorheriger Adapter gefunden — starte OS2-LoRA-Basislauf.')
    lora = LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        bias='none'
    )
    model = get_peft_model(model, lora)
model.print_trainable_parameters()
model.gradient_checkpointing_enable()
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

SYSTEM_TEXT = ('Du bist LiveOS, ein hilfreicher deutschsprachiger KI-Assistent. '
               'Antworte freundlich, klar und ehrlich. Erfinde keine Fakten. '
               'Wenn dir Informationen fehlen, frage nach oder sage offen, dass du es nicht weißt.')
rows = []
for item in examples:
    answer = str(item.get('ideal_response') or '').strip()
    if not answer:
        continue
    for phrase in item.get('trigger_phrases') or []:
        user_text = str(phrase).strip()
        if len(user_text) < 3:
            continue
        if user_text[-1] not in '?!。':
            user_text += '?'
        style = str(item.get('style_guidance') or '').strip()
        system = SYSTEM_TEXT + ((' Stilhinweis: ' + style) if style else '')
        messages = [{'role':'system','content':system},{'role':'user','content':user_text}]
        prefix_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
        full_ids = tokenizer.apply_chat_template(messages + [{'role':'assistant','content':answer}], tokenize=True, add_generation_prompt=False)
        max_len = 768
        full_ids = full_ids[:max_len]
        labels = ([-100] * min(len(prefix_ids), len(full_ids))) + full_ids[len(prefix_ids):]
        labels = labels[:max_len]
        if not any(v != -100 for v in labels):
            continue
        rows.append({'input_ids':full_ids,'attention_mask':[1]*len(full_ids),'labels':labels})
if len(rows) < 3:
    raise RuntimeError('Nach Aufbereitung sind zu wenige Trainingszeilen übrig.')
train_dataset = Dataset.from_list(rows)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100)
print('LoRA-Trainingszeilen:', len(train_dataset))


In [ ]:
output_dir = '/kaggle/working/liveos-os2-checkpoints'
args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    fp16=True,
    gradient_checkpointing=True,
    optim='adamw_torch',
    logging_strategy='steps',
    logging_steps=2,
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
    dataloader_pin_memory=True,
    remove_unused_columns=False
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=collator,
    tokenizer=tokenizer
)
train_result = trainer.train()
training_loss = float(train_result.training_loss) if train_result.training_loss is not None else None
print('Echtes LoRA-Fine-Tuning abgeschlossen. Trainingsverlust:', training_loss)

adapter_dir = '/kaggle/working/liveos-os2-adapter'
model.save_pretrained(adapter_dir, safe_serialization=True)
tokenizer.save_pretrained(adapter_dir)
with open(os.path.join(adapter_dir,'README.md'),'w',encoding='utf-8') as f:
    f.write('# LiveOS OS2 LoRA Adapter\\n\\nPrivate adapter fine-tuned on approved LiveOS OS2 examples. Base model: Qwen/Qwen2.5-0.5B-Instruct (Apache-2.0). This is an adapter, not a standalone foundation model.\\n')
api.upload_folder(
    repo_id=HF_REPO, repo_type='model', folder_path=adapter_dir,
    commit_message='Update LiveOS OS2 LoRA adapter weights'
)
print('Aktualisierte Adapter-Gewichte privat auf Hugging Face gesichert.')


In [ ]:
def generate_answer(prompt):
    messages = [
        {'role':'system','content':SYSTEM_TEXT},
        {'role':'user','content':str(prompt)}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=128, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    new_tokens = output[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

evaluation_rows = []
for i, benchmark in enumerate(benchmarks, start=1):
    candidate = generate_answer(benchmark['prompt'])
    score = rpc('liveos_os2_worker_score_candidate', {
        'p_run_id':run_id,
        'p_benchmark_id':benchmark['id'],
        'p_candidate':candidate
    })
    evaluation_rows.append({'benchmark':benchmark['benchmark_key'], 'score':score.get('points',0), 'passed':score.get('passed',False)})
    print(f"{i}/{len(benchmarks)}: {benchmark['benchmark_key']} → {score.get('points',0)} Punkte")

passed = sum(1 for row in evaluation_rows if row['passed'])
average = sum(float(row['score']) for row in evaluation_rows) / max(1,len(evaluation_rows))
print(f'Hidden Benchmark: {passed}/{len(evaluation_rows)} bestanden; Durchschnitt {average:.2f}/100')


In [ ]:
result = rpc('liveos_os2_worker_finish_run', {
    'p_run_id':run_id,
    'p_training_loss':training_loss,
    'p_model_repo':HF_REPO,
    'p_epochs':2,
    'p_training_examples':len(train_dataset)
})
print('OS2-Lauf in Supabase abgeschlossen:')
print(json.dumps(result, ensure_ascii=False, indent=2))
print('Adapter-Repository: https://huggingface.co/' + HF_REPO)
print('Wichtig: Das Modell wurde trainiert, aber das LiveOS-Chat-Frontend nutzt diesen Adapter noch nicht als Inference-Backend.')
